In [ ]:
#Specify the directory for input data and output pickles to save the model parameters
#see comments for more detail on pickes in the last cell.. 
datadir='./'

#dataset = 1 for KMNIST, 2 for K49
dataset = 2

#Dataset 2(K49) may take longer computation time than one day).
#You may test with small dataset by setting the following variable to 'True'
small_dataset = False

#if the above parameter is set to be 'True', then only the following number of instances will be used for training.
#number of training data in original dataset 1 and 2 is 60000 and 232365 respectively.
number_of_instances_for_small_dataset = 1000 

#multi-class decision function. While an 'ovr' option is fast. an'ovo' optionis very slow, but, it classifties better for unbalanced numbers of instances among classes.  
decision_function_shape="ovr"

# Make sure that the following files are set in 'datadir' directory.

if dataset == 1:
    #Dataset 1 KMNNIST 10
    filename_train_img   = "kmnist-train-imgs"
    filename_train_label = "kmnist-train-labels"
    filename_test_img    = "kmnist-test-imgs"
    filename_test_label  = "kmnist-test-labels"
    pickles_id = "data1"
elif dataset == 2:
    #Dataset 2 K49
    filename_train_img   = "k49-train-imgs"
    filename_train_label = "k49-train-labels"
    filename_test_img    = "k49-test-imgs"
    filename_test_label  = "k49-test-labels"
    pickles_id = "data2"

In [ ]:
import numpy as np
import os

np.random.seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import pickle
from sklearn.svm import SVC
from sklearn import datasets
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import reciprocal, uniform

In [ ]:
################################
### Data Input
################################

X_train = np.load(datadir+filename_train_img+'.npz')['arr_0']
y_train = np.load(datadir+filename_train_label+'.npz')['arr_0']
X_test = np.load(datadir+filename_test_img+'.npz')['arr_0']
y_test = np.load(datadir+filename_test_label+'.npz')['arr_0']


X_train = np.reshape(X_train,(-1,28*28))
X_test  = np.reshape(X_test, (-1,28*28))


ntrain = len(X_train)
ntest = len(X_test)

np.random.seed(42)
rnd_idx = np.random.permutation(ntrain)
X_train = X_train[rnd_idx]
y_train = y_train[rnd_idx]

X_train_rs = X_train[:1000]
y_train_rs = y_train[:1000]

if small_dataset:
    X_train = X_train[:number_of_instances_for_small_dataset]
    y_train = y_train[:number_of_instances_for_small_dataset]

#scaler = StandardScaler()
X_train_scaled = X_train/255
X_test_scaled = X_test/255

X_train_scaled_rs = X_train_rs/255

print (len(X_train),len(y_train))

In [ ]:
################################
### Random Search to find best 'C' and 'gamma'　
### (This process can be skipped. These parameters are given in the next cell.
################################ 

svm_clf = SVC(decision_function_shape=decision_function_shape)
param_distributions = {"gamma": reciprocal(0.001, 0.1), "C": uniform(1, 10)}
rnd_search_cv = RandomizedSearchCV(svm_clf, param_distributions, n_iter=10, verbose=2)
rnd_search_cv.fit(X_train_scaled_rs, y_train_rs)
print (rnd_search_cv.best_estimator_)

In [ ]:
################################
### SVM Definition with given parameters (C, gamma)
################################ 
if dataset == 1:
    #Dataset 1 KMNNIST 10
    C = 10.572971700825061
    gamma = 0.01381957639278581
elif dataset == 2:
    #Dataset 2 K49
    C = 4.152705171689228
    gamma = 0.006783091541660457
    
svm_clf = SVC(C=C, cache_size=200, class_weight=None, coef0=0.0,
  decision_function_shape=decision_function_shape, degree=3, gamma=gamma,
  kernel='rbf', max_iter=-1, probability=False, random_state=None,
  shrinking=True, tol=0.001, verbose=False)

In [ ]:
#SVM Training (fitting)
svm_clf.fit(X_train_scaled, y_train)
print ('fitting done')
with open(datadir+'Kuzushi_SVC_class_after_fitting_'+pickles_id+'.pickle', mode='wb') as f:
    pickle.dump(svm_clf,f)

#SVM Prediction for the training dataset    
y_pred = svm_clf.predict(X_train_scaled)
with open(datadir+'Kuzushi_y_predicted_for_training_data_'+pickles_id+'.pickle', mode='wb') as f2:
    pickle.dump(y_pred,f2)
acc_train=accuracy_score(y_train, y_pred)
print ('accuracy on training data(not class average)',acc_train)

#SVM Prediction for the test dataset
y_pred_test = svm_clf.predict(X_test_scaled)
with open(datadir+'Kuzushi_y_predicted_for_test_data_'+pickles_id+'.pickle', mode='wb') as f3:
    pickle.dump(y_pred_test,f3)
acc_test=accuracy_score(y_test, y_pred_test)
print ('accuracy on test data(not class average)',acc_test)

In [ ]:
if dataset==2:
    accuracy_train = 0
    accuracy_test = 0
    for i in range(0,49):
        y_index_train = np.where(y_train==i)
        y_index_test = np.where(y_test==i)
        acc_train=accuracy_score(y_train[y_index_train], y_pred[y_index_train])
        acc_test=accuracy_score(y_test[y_index_test], y_pred_test[y_index_test])
        accuracy_train+=acc_train/49
        accuracy_test+=acc_test/49
    print ('accuracy on training data(class averaged)',accuracy_train)
    print ('accuracy on test data(class averaged)',accuracy_test)